In [1]:
# 1. 导入库
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, BatchNormalization, Dropout
from keras.regularizers import l2
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
import numpy as np

# 2. 加载数据
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# 3. 数据预处理
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
Y_train = np_utils.to_categorical(y_train, 10)
Y_test = np_utils.to_categorical(y_test, 10)

# 4. 数据增强（移除亮度参数）
datagen = ImageDataGenerator(
    rotation_range=30,          
    width_shift_range=0.2,       
    height_shift_range=0.2,      
    horizontal_flip=True,
    vertical_flip=True,          
    zoom_range=0.2,              
    shear_range=0.2,             
    fill_mode='nearest'
)
datagen.fit(X_train)

# 5. 构建模型
model = Sequential()

# 第一层卷积块
model.add(Conv2D(32, (3,3), padding='same', input_shape=X_train.shape[1:], kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(32, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# 第二层卷积块
model.add(Conv2D(64, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(64, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# 第三层卷积块
model.add(Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# 全连接层
model.add(Flatten())
model.add(Dense(512, kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

# 6. 编译模型
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.0001),
    metrics=['accuracy']
)

# 7. 早停策略
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True
)

# 8. 训练模型
history = model.fit(
    datagen.flow(X_train, Y_train, batch_size=64),
    steps_per_epoch=len(X_train) // 64,
    epochs=50,
    validation_data=(X_test, Y_test),
    callbacks=[early_stopping]
)

# 9. 评估模型
score = model.evaluate(X_test, Y_test)
print(f'Test loss: {score[0]:.4f}')
print(f'Test accuracy: {score[1]:.4f}')

(50000, 32, 32, 3) (10000, 32, 32, 3) (50000, 1) (10000, 1)
Epoch 1/50
781/781 [==============================] - 93s 117ms/step - loss: 2.8729 - accuracy: 0.2264 - val_loss: 2.4314 - val_accuracy: 0.3205
Epoch 2/50
781/781 [==============================] - 95s 121ms/step - loss: 2.5301 - accuracy: 0.2992 - val_loss: 2.1792 - val_accuracy: 0.3884
Epoch 3/50
781/781 [==============================] - 99s 127ms/step - loss: 2.3706 - accuracy: 0.3403 - val_loss: 2.2940 - val_accuracy: 0.3702
Epoch 4/50
781/781 [==============================] - 88s 113ms/step - loss: 2.2588 - accuracy: 0.3703 - val_loss: 2.1793 - val_accuracy: 0.4029
Epoch 5/50
781/781 [==============================] - 73s 94ms/step - loss: 2.1650 - accuracy: 0.3970 - val_loss: 2.1234 - val_accuracy: 0.4152
Epoch 6/50
781/781 [==============================] - 90s 116ms/step - loss: 2.0860 - accuracy: 0.4155 - val_loss: 1.9855 - val_accuracy: 0.4473
Epoch 7/50
781/781 [==============================] - 97s 124ms/step - 